# 📱 Phân Loại Tin Nhắn Safe / Smishing
### Sử dụng TF-IDF + Logistic Regression

**Pipeline:**
1. Load & khám phá dữ liệu (EDA)
2. Tiền xử lý văn bản tiếng Việt
3. Vector hóa bằng TF-IDF
4. Huấn luyện Logistic Regression
5. Đánh giá mô hình
6. Dự đoán tin nhắn mới

## 0. Cài đặt thư viện

In [ ]:
# Cài đặt underthesea để xử lý tiếng Việt
!pip install underthesea -q

## 1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Visualize
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import seaborn as sns
from wordcloud import WordCloud

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc,
    accuracy_score, f1_score
)
from sklearn.preprocessing import LabelEncoder

# Tiếng Việt
try:
    from underthesea import word_tokenize
    USE_UNDERTHESEA = True
    print('✅ underthesea loaded - dùng word_tokenize tiếng Việt')
except ImportError:
    USE_UNDERTHESEA = False
    print('⚠️  underthesea chưa cài - dùng tokenize đơn giản')

print('✅ Import thành công!')

## 2. Load Dataset

In [ ]:
# ===================================================
# 👉 THAY ĐỔI ĐƯỜNG DẪN FILE CSV CỦA BẠN Ở ĐÂY
# ===================================================
CSV_PATH = 'your_dataset.csv'   # <-- sửa thành tên file CSV của bạn
CONTENT_COL = 'content'         # tên cột văn bản
LABEL_COL   = 'label'           # tên cột nhãn (safe / smishing)

df = pd.read_csv(CSV_PATH)
print(f'📂 Dataset shape: {df.shape}')
df.head()

## 3. Khám phá dữ liệu (EDA)

In [ ]:
# Thông tin cơ bản
print('=== Thông tin dataset ===')
print(df.info())
print('\n=== Giá trị null ===')
print(df[[CONTENT_COL, LABEL_COL]].isnull().sum())

In [ ]:
# Phân phối nhãn
label_counts = df[LABEL_COL].value_counts()
print('=== Phân phối nhãn ===')
print(label_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
label_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Số lượng mẫu theo nhãn', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Nhãn')
axes[0].set_ylabel('Số lượng')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(int(bar.get_height())), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Tỷ lệ phân phối nhãn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Phân phối độ dài tin nhắn
df['text_length'] = df[CONTENT_COL].astype(str).apply(len)
df['word_count']  = df[CONTENT_COL].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color in zip(df[LABEL_COL].unique(), colors):
    subset = df[df[LABEL_COL] == label]
    axes[0].hist(subset['text_length'], bins=40, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['word_count'],  bins=40, alpha=0.6, label=label, color=color)

axes[0].set_title('Phân phối độ dài ký tự', fontweight='bold')
axes[0].set_xlabel('Số ký tự'); axes[0].legend()

axes[1].set_title('Phân phối số từ', fontweight='bold')
axes[1].set_xlabel('Số từ'); axes[1].legend()

plt.tight_layout()
plt.savefig('text_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby(LABEL_COL)[['text_length', 'word_count']].describe().round(1))

## 4. Tiền xử lý văn bản tiếng Việt

In [ ]:
# Stopwords tiếng Việt phổ biến
VIETNAMESE_STOPWORDS = set([
    'và', 'của', 'có', 'được', 'trong', 'là', 'các', 'cho', 'với',
    'không', 'này', 'đã', 'những', 'theo', 'trên', 'từ', 'về', 'tại',
    'bị', 'người', 'đến', 'như', 'khi', 'hay', 'thì', 'mà', 'ra',
    'cũng', 'đó', 'vào', 'một', 'để', 'lên', 'còn', 'vì', 'sau',
    'đây', 'nên', 'sẽ', 'nhiều', 'lại', 'rất', 'hơn', 'đang', 'sẽ',
    'bạn', 'tôi', 'chúng', 'họ', 'đều', 'thể', 'nhưng', 'nào', 'tới',
    'giữa', 'trước', 'vẫn', 'ai', 'ta', 'mình', 'chỉ', 'bởi', 'hãy'
])

def preprocess_text(text):
    """Tiền xử lý văn bản tiếng Việt."""
    if not isinstance(text, str):
        return ''
    
    # Lowercase
    text = text.lower()
    
    # Chuẩn hóa số điện thoại, URL, email
    text = re.sub(r'http\S+|www\.\S+', ' url ', text)
    text = re.sub(r'\S+@\S+', ' email ', text)
    text = re.sub(r'(\+84|0)\d{9,10}', ' sodienthoai ', text)
    
    # Giữ chữ cái (Việt + Latin) và khoảng trắng
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', ' ', text)
    
    # Xóa số thuần túy
    text = re.sub(r'\b\d+\b', ' ', text)
    
    # Tokenize
    if USE_UNDERTHESEA:
        tokens = word_tokenize(text, format='text').split()
    else:
        tokens = text.split()
    
    # Loại stopwords và token quá ngắn
    tokens = [t for t in tokens if t not in VIETNAMESE_STOPWORDS and len(t) > 1]
    
    return ' '.join(tokens)

print('🔄 Đang xử lý văn bản...')
df['content_clean'] = df[CONTENT_COL].apply(preprocess_text)
print('✅ Xong!')

# Xem kết quả
print('\n=== Ví dụ trước/sau tiền xử lý ===')
for i in range(3):
    print(f'\n[{i+1}] Gốc   : {df[CONTENT_COL].iloc[i][:100]}')
    print(f'     Sạch  : {df["content_clean"].iloc[i][:100]}')

## 5. Chia tập Train / Test

In [ ]:
# Loại bỏ mẫu rỗng sau tiền xử lý
df_clean = df[df['content_clean'].str.strip() != ''].copy()
print(f'Sau khi lọc: {len(df_clean)} mẫu (bỏ {len(df) - len(df_clean)} mẫu rỗng)')

X = df_clean['content_clean']
y = df_clean[LABEL_COL]

# Encode nhãn nếu không phải số
if y.dtype == object:
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print(f'\nEncoding nhãn: {dict(zip(le.classes_, le.transform(le.classes_)))}')
else:
    y_encoded = y.values
    le = None

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'\n📊 Tập huấn luyện : {len(X_train)} mẫu')
print(f'📊 Tập kiểm tra   : {len(X_test)} mẫu')

## 6. Xây dựng Pipeline TF-IDF + Logistic Regression

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),     # unigram + bigram
        max_features=30000,
        min_df=2,               # bỏ từ xuất hiện < 2 lần
        max_df=0.95,            # bỏ từ xuất hiện > 95% tài liệu
        sublinear_tf=True,      # log(TF) để giảm dominance
    )),
    ('clf', LogisticRegression(
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced', # xử lý mất cân bằng nhãn
        random_state=42
    ))
])

print('Pipeline:')
print(pipeline)

## 7. Huấn luyện mô hình

In [ ]:
print('🚀 Đang huấn luyện...')
pipeline.fit(X_train, y_train)
print('✅ Huấn luyện xong!')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_weighted')
print(f'\n📈 Cross-validation F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'   Từng fold: {[f"{s:.4f}" for s in cv_scores]}')

## 8. Đánh giá mô hình

In [ ]:
y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

label_names = le.classes_.tolist() if le else ['safe', 'smishing']

print('=' * 55)
print('           CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🎯 Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'🎯 F1 Score : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'🎯 AUC-ROC  : {roc_auc:.4f}')

## 9. Phân tích đặc trưng quan trọng

In [ ]:
# Top từ quan trọng nhất theo Logistic Regression
tfidf_vectorizer = pipeline.named_steps['tfidf']
lr_model         = pipeline.named_steps['clf']
feature_names    = tfidf_vectorizer.get_feature_names_out()
coefficients     = lr_model.coef_[0]

top_n = 20
top_smishing_idx = np.argsort(coefficients)[::-1][:top_n]
top_safe_idx     = np.argsort(coefficients)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Smishing keywords
smishing_words  = [feature_names[i] for i in top_smishing_idx]
smishing_scores = [coefficients[i]  for i in top_smishing_idx]
axes[0].barh(smishing_words[::-1], smishing_scores[::-1], color='#e74c3c', edgecolor='black')
axes[0].set_title(f'Top {top_n} từ → SMISHING', fontsize=12, fontweight='bold', color='#e74c3c')
axes[0].set_xlabel('Hệ số Logistic Regression')

# Safe keywords
safe_words  = [feature_names[i] for i in top_safe_idx]
safe_scores = [abs(coefficients[i]) for i in top_safe_idx]
axes[1].barh(safe_words[::-1], safe_scores[::-1], color='#2ecc71', edgecolor='black')
axes[1].set_title(f'Top {top_n} từ → SAFE', fontsize=12, fontweight='bold', color='#27ae60')
axes[1].set_xlabel('|Hệ số Logistic Regression|')

plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Word Cloud
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, label_val, title, color in zip(
    axes,
    [0, 1] if le is None else le.transform(le.classes_),
    label_names,
    ['Greens', 'Reds']
):
    texts = ' '.join(df_clean[df_clean[LABEL_COL] == (label_val if le is None else le.inverse_transform([label_val])[0])]['content_clean'])
    if texts.strip():
        wc = WordCloud(width=700, height=350, background_color='white',
                       colormap=color, max_words=100).generate(texts)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud: {title.upper()}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Phân tích lỗi

In [ ]:
X_test_list = X_test.tolist()
errors = [
    {
        'content'   : X_test_list[i],
        'true_label': label_names[y_test[i]],
        'pred_label': label_names[y_pred[i]],
        'confidence': max(pipeline.predict_proba([X_test_list[i]])[0])
    }
    for i in range(len(y_test)) if y_test[i] != y_pred[i]
]

errors_df = pd.DataFrame(errors).sort_values('confidence', ascending=False)
print(f'Tổng số lỗi: {len(errors_df)} / {len(y_test)} ({len(errors_df)/len(y_test)*100:.1f}%)')
print('\n=== Các mẫu dự đoán sai (confidence cao nhất) ===')
errors_df.head(10)

## 11. Dự đoán tin nhắn mới

In [ ]:
def predict_message(message, threshold=0.5):
    """Dự đoán một tin nhắn mới."""
    cleaned   = preprocess_text(message)
    prob      = pipeline.predict_proba([cleaned])[0]
    pred_idx  = int(prob[1] >= threshold)
    pred_label = label_names[pred_idx]
    
    icon = '🚨' if pred_label.lower() == 'smishing' else '✅'
    print(f'{icon} Tin nhắn : {message[:80]}...' if len(message) > 80 else f'{icon} Tin nhắn : {message}')
    print(f'   Nhãn dự đoán : {pred_label.upper()}')
    print(f'   Xác suất safe={prob[0]:.4f} | smishing={prob[1]:.4f}')
    print()
    return pred_label, prob

# ===================================
# 👉 THỬ DỰ ĐOÁN TIN NHẮN MỚI
# ===================================
test_messages = [
    "Tài khoản ngân hàng của bạn có vấn đề. Vui lòng click vào link để xác nhận: http://fake-bank.com/verify",
    "Chúc mừng! Bạn đã trúng thưởng 50 triệu đồng. Liên hệ ngay: 0909123456",
    "Nhớ mua rau về nhé, mình về muộn một chút",
    "Cuộc họp ngày mai dời sang 3 giờ chiều nhé anh",
]

for msg in test_messages:
    predict_message(msg)

## 12. Lưu mô hình

In [ ]:
import joblib

joblib.dump(pipeline, 'smishing_model.joblib')
if le:
    joblib.dump(le, 'label_encoder.joblib')

print('✅ Đã lưu mô hình: smishing_model.joblib')
if le:
    print('✅ Đã lưu label encoder: label_encoder.joblib')

# Load lại và kiểm tra
loaded_pipeline = joblib.load('smishing_model.joblib')
test_pred = loaded_pipeline.predict([preprocess_text(test_messages[0])])
print(f'\n🔄 Kiểm tra load lại: dự đoán = {label_names[test_pred[0]]}')

## 13. Tóm tắt kết quả

In [ ]:
print('=' * 55)
print('           TỔNG KẾT MÔ HÌNH')
print('=' * 55)
print(f'  Dataset          : {len(df_clean)} mẫu')
print(f'  Train / Test     : {len(X_train)} / {len(X_test)}')
print(f'  TF-IDF features  : {tfidf_vectorizer.get_feature_names_out().shape[0]:,}')
print(f'  Accuracy         : {accuracy_score(y_test, y_pred):.4f}')
print(f'  F1 (weighted)    : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'  AUC-ROC          : {roc_auc:.4f}')
print(f'  CV F1 (5-fold)   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 55)